In [1]:
import jax
import jax.numpy as jnp
from flax import linen as nn

from metanca.nn import forward_factory, flatten_params
from metanca.neighbors import build_compute_graph, build_parameter_neighbor_graphs, ParamCells

In [2]:
class TinyMLP(nn.Module):
    hidden_dim: int = 8
    out_dim: int = 2

    @nn.compact
    def __call__(self, x: jax.Array) -> jax.Array:
        dense1 = nn.Dense(self.hidden_dim)
        dense2 = nn.Dense(self.out_dim)
        h = dense1(x)
        act = nn.relu(h)
        return dense2(act)

class ResidualMLP(nn.Module):
    n_layers: int = 2
    hidden_dim: int = 8
    out_dim: int = 2

    @nn.compact
    def __call__(self, x):
        
        residual = TinyMLP(self.hidden_dim, self.hidden_dim)(x)        
        for i in range(self.n_layers - 1):
            y = TinyMLP(self.hidden_dim, self.hidden_dim)(residual)
            residual = residual + y
                
        # Assume x.shape[-1] == out_dim
        return residual

In [3]:
key = jax.random.key(0)
x = jnp.ones((1, 4))
model = ResidualMLP(n_layers=3)
params = model.init(key, x)
forward = forward_factory(model, params)
jaxpr = jax.make_jaxpr(forward)(x).jaxpr

In [4]:
vars_and_params = [(var, param, name) for var, (name, param) in zip(jaxpr.constvars, flatten_params(params["params"]))]
compute_graph = build_compute_graph(jaxpr)
fwd, bwd = build_parameter_neighbor_graphs(compute_graph)

In [5]:
cells = ParamCells(vars_and_params, fwd, bwd)

In [6]:
for name in cells.get_param_names():
    bwd_neighbors = cells.get_backward_neighbors(name)
    print(f"backward neighbors of {name}\n\t{[(n, vecs.shape) for n, vecs in bwd_neighbors]}")

backward neighbors of TinyMLP_0.Dense_0.bias
	[]
backward neighbors of TinyMLP_0.Dense_0.kernel
	[]
backward neighbors of TinyMLP_0.Dense_1.bias
	[]
backward neighbors of TinyMLP_0.Dense_1.kernel
	[('TinyMLP_0.Dense_0.kernel', (4, 8, 17)), ('TinyMLP_0.Dense_0.bias', (1, 8, 17))]
backward neighbors of TinyMLP_1.Dense_0.bias
	[]
backward neighbors of TinyMLP_1.Dense_0.kernel
	[('TinyMLP_0.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_0.Dense_1.bias', (1, 8, 17))]
backward neighbors of TinyMLP_1.Dense_1.bias
	[]
backward neighbors of TinyMLP_1.Dense_1.kernel
	[('TinyMLP_1.Dense_0.bias', (1, 8, 17)), ('TinyMLP_1.Dense_0.kernel', (8, 8, 17))]
backward neighbors of TinyMLP_2.Dense_0.bias
	[]
backward neighbors of TinyMLP_2.Dense_0.kernel
	[('TinyMLP_0.Dense_1.bias', (1, 8, 17)), ('TinyMLP_0.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_1.Dense_1.bias', (1, 8, 17)), ('TinyMLP_1.Dense_1.kernel', (8, 8, 17))]
backward neighbors of TinyMLP_2.Dense_1.bias
	[]
backward neighbors of TinyMLP_2.Dense_1.kernel
	[

In [9]:
forward_neighbors = cells.get_forward_neighbors("TinyMLP_1.Dense_0.kernel")

In [11]:
[text for text, _ in forward_neighbors]

['TinyMLP_1.Dense_0.kernel',
 'TinyMLP_1.Dense_0.bias',
 'TinyMLP_1.Dense_1.kernel']

In [13]:
forward_neighbors[0][1].shape

(8, 8, 17)

In [8]:
for name in cells.get_param_names():
    fwd_neighbors = cells.get_forward_neighbors(name)
    print(f"forward neighbors of {name}\n\t{[(n, vecs.shape) for n, vecs in fwd_neighbors]}")

forward neighbors of TinyMLP_0.Dense_0.bias
	[('TinyMLP_0.Dense_0.bias', (8, 1, 17)), ('TinyMLP_0.Dense_0.kernel', (4, 8, 17)), ('TinyMLP_0.Dense_1.kernel', (8, 8, 17))]
forward neighbors of TinyMLP_0.Dense_0.kernel
	[('TinyMLP_0.Dense_0.kernel', (8, 4, 17)), ('TinyMLP_0.Dense_0.bias', (8, 1, 17)), ('TinyMLP_0.Dense_1.kernel', (8, 8, 17))]
forward neighbors of TinyMLP_0.Dense_1.bias
	[('TinyMLP_0.Dense_1.bias', (8, 1, 17)), ('TinyMLP_2.Dense_0.kernel', (8, 8, 17)), ('TinyMLP_2.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_0.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_2.Dense_1.bias', (8, 1, 17)), ('TinyMLP_1.Dense_1.bias', (8, 1, 17)), ('TinyMLP_1.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_1.Dense_0.kernel', (8, 8, 17))]
forward neighbors of TinyMLP_0.Dense_1.kernel
	[('TinyMLP_0.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_2.Dense_0.kernel', (8, 8, 17)), ('TinyMLP_0.Dense_1.bias', (8, 1, 17)), ('TinyMLP_2.Dense_1.kernel', (8, 8, 17)), ('TinyMLP_2.Dense_1.bias', (8, 1, 17)), ('TinyMLP_1.Dense_1.bias', (